## Hybrid Search 

- We don't use Reciprocal Rank Fusion (RRF) in Pinecone, it natively implements native hybrid scoring
- RRF is used in other vectorstores for hybrid search when you perform two completely separate searches.
- RRF Used in : Elasticsearch, Weaviate, Vespa, Multi-vector retrieval, Ensemble retrievers, etc

In [8]:
import os
import uuid
from dotenv import load_dotenv
from langchain_ollama import OllamaEmbeddings, ChatOllama
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder # it uses TF-IDF under the hood for sparse matrix 

In [9]:
load_dotenv()
pinecone_api_key = os.getenv('PINECONE_API_KEY')

In [10]:
index_name = "hybrid-search-langchain-pinecone"

# Initialize Pinecone
pc = Pinecone(api_key=pinecone_api_key)

# Create index if it doesn't exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        index_name, 
        dimension=384,  # dimension for dense vector
        metric="dotproduct",  # sparse values supported for dotproduct
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    ) 

index_obj = pc.Index(index_name)

In [11]:
# Dense Vector and Sparse matrix
embedding = OllamaEmbeddings(model="qwen3-embedding:4b", dimensions=384)
encoder = BM25Encoder().default()

In [12]:
texts = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
]

# Dense embeddings
dense_vectors = embedding.embed_documents(texts)

# Sparse embeddings (BM25)
sparse_vectors = encoder.encode_documents(texts)

vectors = []

for text, dense, sparse in zip(texts, dense_vectors, sparse_vectors):
    vectors.append(
        {
            "id": str(uuid.uuid4()),
            "values": dense,
            "sparse_values": sparse,
            "metadata": {
                "text": text
            }
        }
    )

index_obj.upsert(vectors=vectors)

{'upserted_count': 3}

In [13]:
query = "What city did I visit last?"

dense_query = embedding.embed_query(query)
sparse_query = encoder.encode_queries(query)

results = index_obj.query(
    vector=dense_query,
    sparse_vector=sparse_query,
    top_k=3,
    include_metadata=True
)

for match in results["matches"]:
    print(match["metadata"]["text"])
    print(match["score"])
    print("-" * 50)

In 2022, I visited New York
0.965649366
--------------------------------------------------
In 2023, I visited Paris
0.949121356
--------------------------------------------------
In 2021, I visited New Orleans
0.928804398
--------------------------------------------------
